#  Fine-Tuning XLSR-Wav2Vec2 for Multilingual ASR

This technique of TTS focuses on adapting a pretrained XLSR-Wav2Vec2 model for Automatic Speech Recognition (ASR) using the  Transformers library.

We leveraged transfer learning to fine-tune a multilingual self-supervised speech model on labeled audio data and evaluate performance using Word Error Rate (WER).

The goal is to build an efficient, scalable ASR system for low-resource language speech recognition.

##  Environment Setup

In this step, we install all required dependencies for building and training the ASR pipeline.

- `datasets` – For loading and preprocessing speech datasets  
- `transformers` – For pretrained XLSR-Wav2Vec2 and Trainer API  
- `torchaudio` & `librosa` – For audio processing  
- `jiwer` – For evaluating Word Error Rate (WER)

The `%%capture` magic command suppresses verbose installation logs to keep the notebook clean.

In [ ]:
%%capture
!pip install datasets==1.4.1
!pip install transformers==4.4.0
!pip install torchaudio
!pip install librosa
!pip install jiwer

##  Loading the Punjabi Speech Dataset

We load the Punjabi (pa-IN) subset of the Mozilla Common Voice dataset using the Hugging Face `datasets` library.

- `train+validation` split is combined for model training  
- `test` split is reserved for final evaluation  

This dataset provides paired audio recordings and corresponding text transcripts required for supervised ASR fine-tuning.

In [ ]:
from datasets import load_dataset, load_metric

common_voice_train = load_dataset("common_voice", "pa-IN", split="train+validation")
common_voice_test = load_dataset("common_voice", "pa-IN", split="test")

##  Dataset Cleanup

We remove non-essential metadata columns such as speaker details and voting information.

Only the required fields are retained:
- `audio`
- `sentence` (transcript)

This reduces memory usage and ensures the dataset contains only information relevant for ASR training.

In [ ]:
common_voice_train = common_voice_train.remove_columns(["accent", "age", "client_id", "down_votes", "gender", "locale", "segment", "up_votes"])
common_voice_test = common_voice_test.remove_columns(["accent", "age", "client_id", "down_votes", "gender", "locale", "segment", "up_votes"])

##  Exploring the Dataset

To better understand the structure of our data, we define a helper function that randomly samples examples from the dataset and displays them in a tabular format.

This allows us to:
- Inspect transcripts
- Verify data integrity
- Understand dataset structure before preprocessing

Performing early data inspection helps prevent downstream training issues.

In [ ]:
from datasets import ClassLabel
import random
import pandas as pd
from IPython.display import display, HTML

def show_random_elements(dataset, num_examples=10):
    assert num_examples <= len(dataset), "Can't pick more elements than there are in the dataset."
    picks = []
    for _ in range(num_examples):
        pick = random.randint(0, len(dataset)-1)
        while pick in picks:
            pick = random.randint(0, len(dataset)-1)
        picks.append(pick)

    df = pd.DataFrame(dataset[picks])
    display(HTML(df.to_html()))

##  Sampling Training Examples

We randomly sample 20 examples from the training dataset (excluding the audio file path) to visually inspect the transcripts.

This helps verify:
- Text formatting consistency  
- Language correctness  
- Dataset quality before preprocessing  

Quick sanity checks like this are essential before model training.

In [ ]:
show_random_elements(common_voice_train.remove_columns(["path"]), num_examples=20)

## Text Normalization

We define a preprocessing function to clean transcript text by:

- Removing punctuation and special characters  
- Converting all text to lowercase  
- Appending a trailing space for proper word separation  

Text normalization ensures consistent vocabulary mapping and reduces unnecessary complexity for the model during training.

In [ ]:
import re
chars_to_ignore_regex = '[\,\?\.\!\-\;\:\"\“\%\‘\”\�]'

def remove_special_characters(batch):
    batch["sentence"] = re.sub(chars_to_ignore_regex, '', batch["sentence"]).lower() + " "
    return batch

##  Applying Text Preprocessing

We apply the text normalization function across both the training and test datasets.

This ensures:
- Uniform transcript formatting  
- Cleaner vocabulary construction  
- Reduced noise during model learning  

Consistent preprocessing across splits is essential for reliable evaluation.

In [ ]:
common_voice_train = common_voice_train.map(remove_special_characters)
common_voice_test = common_voice_test.map(remove_special_characters)

##  Verifying Preprocessed Transcripts

After normalization, we again sample random training examples to confirm that:

- Special characters have been removed  
- Text is properly lowercased  
- Formatting is consistent  

Re-checking the data after preprocessing helps ensure transformations were applied correctly before moving forward.

In [ ]:
show_random_elements(common_voice_train.remove_columns(["path"]))

##  Extracting Vocabulary Characters

We define a function to collect all unique characters from the dataset transcripts.

Steps performed:
- Combine all sentences into a single string  
- Extract unique characters  
- Prepare them for vocabulary construction  

This allows us to build a character-level tokenizer tailored specifically to the Punjabi dataset.

In [ ]:
def extract_all_chars(batch):
  all_text = " ".join(batch["sentence"])
  vocab = list(set(all_text))
  return {"vocab": [vocab], "all_text": [all_text]}

##  Building Dataset-Level Vocabulary

We apply the vocabulary extraction function across the entire training and test datasets.

Key configurations:
- `batched=True` and `batch_size=-1` process the full dataset at once  
- `keep_in_memory=True` speeds up processing  
- Original columns are removed to retain only vocabulary data  

This step gathers all unique characters required to construct a complete character-level tokenizer.

In [ ]:
vocab_train = common_voice_train.map(extract_all_chars, batched=True, batch_size=-1, keep_in_memory=True, remove_columns=common_voice_train.column_names)
vocab_test = common_voice_test.map(extract_all_chars, batched=True, batch_size=-1, keep_in_memory=True, remove_columns=common_voice_test.column_names)

## Merging Train and Test Vocabulary

We combine the unique characters extracted from both the training and test datasets.

This ensures:
- Complete character coverage  
- No unseen tokens during evaluation  
- A unified vocabulary for tokenizer creation  

Having a consistent vocabulary across splits is critical for stable ASR performance.

In [ ]:
vocab_list = list(set(vocab_train["vocab"][0]) | set(vocab_test["vocab"][0]))

##  Creating Character-to-Index Mapping

We convert the vocabulary list into a dictionary that maps each character to a unique integer ID.

This mapping:
- Defines the model’s output label space  
- Enables text → numerical token conversion  
- Prepares the vocabulary for CTC-based training  

Each character now has a fixed index used during model learning.

In [ ]:
vocab_dict = {v: k for k, v in enumerate(vocab_list)}
vocab_dict

##  Replacing Space with Word Delimiter Token

We replace the standard space `" "` with a dedicated word delimiter token `"|"`.

Why?

CTC-based models treat the delimiter token as a word boundary indicator during decoding.  
Using `"|"` instead of a raw space ensures cleaner alignment and more reliable transcription output.

This step prepares the vocabulary for proper CTC decoding behavior.

In [ ]:
vocab_dict["|"] = vocab_dict[" "]
del vocab_dict[" "]

##  Adding Special Tokens

We add two essential special tokens to the vocabulary:

- `[UNK]` — Represents unknown or out-of-vocabulary characters  
- `[PAD]` — Used for sequence padding during batching  

These tokens are required for:
- Robust text encoding  
- Proper batch processing  
- Stable CTC training  

Finally, we verify the total vocabulary size, which defines the model’s output dimension.

In [ ]:
vocab_dict["[UNK]"] = len(vocab_dict)
vocab_dict["[PAD]"] = len(vocab_dict)
len(vocab_dict)

Cool, now our vocabulary is complete and consists of 39 tokens, which means that the linear layer that we will add on top of the pretrained XLSR-Wav2Vec2 checkpoint will have an output dimension of 39.

Let's now save the vocabulary as a json file.

In [ ]:
import json
with open('vocab.json', 'w') as vocab_file:
    json.dump(vocab_dict, vocab_file)

In a final step, we use the json file to instantiate an object of the `Wav2Vec2CTCTokenizer` class.

In [ ]:
from transformers import Wav2Vec2CTCTokenizer

tokenizer = Wav2Vec2CTCTokenizer("./vocab.json", unk_token="[UNK]", pad_token="[PAD]", word_delimiter_token="|")

Next, we will create the feature extractor.

##  Initializing the Feature Extractor

We initialize the `Wav2Vec2FeatureExtractor` to process raw audio waveforms before feeding them into the model.

Configuration details:
- `sampling_rate=16000` ensures consistency with the pretrained model  
- `do_normalize=True` scales audio inputs for stable training  
- `padding_value=0.0` enables proper batching  
- `return_attention_mask=True` helps the model ignore padded regions  

This component converts raw speech into normalized numerical inputs compatible with XLSR-Wav2Vec2.

In [ ]:
from transformers import Wav2Vec2FeatureExtractor

feature_extractor = Wav2Vec2FeatureExtractor(feature_size=1, sampling_rate=16000, padding_value=0.0, do_normalize=True, return_attention_mask=True)

## Creating the Unified Processor

We combine the feature extractor (audio preprocessing) and tokenizer (text encoding) into a single `Wav2Vec2Processor`.

This unified processor:
- Converts raw audio → model-ready input values  
- Converts transcripts → token IDs  
- Simplifies both training and inference workflows  

Using a single processor ensures consistency between data preprocessing and model decoding.

In [ ]:
from transformers import Wav2Vec2Processor

processor = Wav2Vec2Processor(feature_extractor=feature_extractor, tokenizer=tokenizer)

##  Mounting Google Drive

We mount Google Drive to enable:

- Saving trained model checkpoints  
- Storing vocabulary files  
- Persisting outputs beyond the runtime session  

This ensures training artifacts are securely stored and accessible after the notebook session ends.

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive/')

## Saving the Processor

We save the `Wav2Vec2Processor` (feature extractor + tokenizer) to Google Drive.

This ensures:
- Reproducibility of preprocessing steps  
- Consistent inference later  
- Easy model reloading without rebuilding the tokenizer  

Saving the processor is essential since the model depends on the exact same vocabulary and audio preprocessing configuration.

In [ ]:
processor.save_pretrained("/content/gdrive/MyDrive/wav2vec2-large-xlsr-pa-IN")

Next, we can prepare the dataset.

##  Inspecting a Sample Data Point

We view the first example from the training dataset to examine its structure.

This helps confirm:
- Available fields (audio, sentence, etc.)
- Data formatting
- Readiness for feature extraction

Understanding the raw data format ensures correct preprocessing in the next steps.

In [ ]:
common_voice_train[0]

## Converting Audio Files to Waveform Arrays

We define a function to:

- Load raw audio files using `torchaudio`
- Convert the waveform into a NumPy array
- Store the sampling rate
- Copy the transcript into a `target_text` field

This step transforms file paths into numerical speech representations, preparing the dataset for feature extraction and model input.

In [ ]:
import torchaudio

def speech_file_to_array_fn(batch):
    speech_array, sampling_rate = torchaudio.load(batch["path"])
    batch["speech"] = speech_array[0].numpy()
    batch["sampling_rate"] = sampling_rate
    batch["target_text"] = batch["sentence"]
    return batch

##  Applying Audio Processing to Dataset

We apply the audio loading function across both training and test datasets.

During this step:
- Audio file paths are converted into waveform arrays  
- Sampling rates are stored  
- Transcripts are preserved as target text  
- Original unused columns are removed  

The dataset now contains structured audio and text data ready for feature extraction.

In [ ]:
common_voice_train = common_voice_train.map(speech_file_to_array_fn, remove_columns=common_voice_train.column_names)
common_voice_test = common_voice_test.map(speech_file_to_array_fn, remove_columns=common_voice_test.column_names)

Great, now we've successfully read in all the audio files, but since we know that Common Voice is sampled at 48kHz, we need to resample the audio files to 16kHz.

Let's make use of the [`librosa`](https://github.com/librosa/librosa) library to downsample the data.

In [ ]:
import librosa
import numpy as np

def resample(batch):
    batch["speech"] = librosa.resample(np.asarray(batch["speech"]), 48_000, 16_000)
    batch["sampling_rate"] = 16_000
    return batch

##  Resampling Audio to 16kHz

We resample all audio files to 16kHz using parallel processing (`num_proc=4`) for efficiency.

Why 16kHz?

The pretrained XLSR-Wav2Vec2 model expects audio sampled at 16kHz.  
Ensuring a consistent sampling rate prevents input mismatch and maintains model compatibility.

Parallel processing speeds up dataset transformation.

In [ ]:
common_voice_train = common_voice_train.map(resample, num_proc=4)
common_voice_test = common_voice_test.map(resample, num_proc=4)

##  Auditing a Random Audio Sample

We randomly select and play a training audio sample to manually verify:

- Audio clarity  
- Proper resampling to 16kHz  
- Correct waveform extraction  

Listening to samples helps detect potential issues such as noise, silence, or corrupted files before model training.

In [ ]:
import IPython.display as ipd
import numpy as np
import random

rand_int = random.randint(0, len(common_voice_train)-1)

ipd.Audio(data=np.asarray(common_voice_train[rand_int]["speech"]), autoplay=True, rate=16000)

##  Inspecting Audio-Text Pair Details

We randomly inspect a sample to verify:

- The target transcript (`target_text`)  
- The shape of the waveform array  
- The sampling rate  

This confirms that:
- Audio is properly loaded  
- Waveform dimensions are valid  
- Sampling rate is consistent  

Validating these details ensures the dataset is correctly prepared before feature extraction and model training.

In [ ]:
rand_int = random.randint(0, len(common_voice_train)-1)

print("Target text:", common_voice_train[rand_int]["target_text"])
print("Input array shape:", np.asarray(common_voice_train[rand_int]["speech"]).shape)
print("Sampling rate:", common_voice_train[rand_int]["sampling_rate"])

##  Preparing Dataset for Model Training

We define a preprocessing function that converts raw audio-text pairs into model-ready inputs.

This function:

- Verifies consistent sampling rate across the batch  
- Converts waveform arrays into `input_values` using the processor  
- Encodes transcripts into numerical `labels`  
- Uses `as_target_processor()` to correctly tokenize target text for CTC training  

After this step, each dataset entry contains:
- `input_values` → audio features for the model  
- `labels` → tokenized transcripts for supervision  

This bridges raw data and the neural network training pipeline.

In [ ]:
def prepare_dataset(batch):
    # check that all files have the correct sampling rate
    assert (
        len(set(batch["sampling_rate"])) == 1
    ), f"Make sure all inputs have the same sampling rate of {processor.feature_extractor.sampling_rate}."

    batch["input_values"] = processor(batch["speech"], sampling_rate=batch["sampling_rate"][0]).input_values

    with processor.as_target_processor():
        batch["labels"] = processor(batch["target_text"]).input_ids
    return batch

##  Applying Final Dataset Preparation

We apply the `prepare_dataset` function across both training and test splits using batched and parallel processing.

Configuration highlights:
- `batched=True` enables batch-level processing  
- `batch_size=8` balances memory and speed  
- `num_proc=4` accelerates preprocessing  
- Original columns are removed to retain only model inputs  

After this step, the dataset contains:
- `input_values` → processed audio features  
- `labels` → tokenized transcripts  

The data is now fully prepared for model fine-tuning.

In [ ]:
common_voice_train = common_voice_train.map(prepare_dataset, remove_columns=common_voice_train.column_names, batch_size=8, num_proc=4, batched=True)
common_voice_test = common_voice_test.map(prepare_dataset, remove_columns=common_voice_test.column_names, batch_size=8, num_proc=4, batched=True)

##  Custom Data Collator for CTC Training

We implement a custom `DataCollatorCTCWithPadding` to dynamically pad audio inputs and labels during batching.

Why is this necessary?

- Audio sequences have variable lengths  
- Transcripts have variable token lengths  
- CTC requires correct padding handling  

This collator:

- Separately pads `input_values` (audio features)  
- Separately pads `labels` (tokenized transcripts)  
- Replaces label padding with `-100` so it is ignored during CTC loss computation  
- Returns PyTorch tensors ready for training  

Dynamic padding ensures efficient batching while maintaining correct loss behavior for sequence-to-sequence speech modeling.

In [ ]:
import torch

from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional, Union

@dataclass
class DataCollatorCTCWithPadding:
    """
    Data collator that will dynamically pad the inputs received.
    Args:
        processor (:class:`~transformers.Wav2Vec2Processor`)
            The processor used for proccessing the data.
        padding (:obj:`bool`, :obj:`str` or :class:`~transformers.tokenization_utils_base.PaddingStrategy`, `optional`, defaults to :obj:`True`):
            Select a strategy to pad the returned sequences (according to the model's padding side and padding index)
            among:
            * :obj:`True` or :obj:`'longest'`: Pad to the longest sequence in the batch (or no padding if only a single
              sequence if provided).
            * :obj:`'max_length'`: Pad to a maximum length specified with the argument :obj:`max_length` or to the
              maximum acceptable input length for the model if that argument is not provided.
            * :obj:`False` or :obj:`'do_not_pad'` (default): No padding (i.e., can output a batch with sequences of
              different lengths).
        max_length (:obj:`int`, `optional`):
            Maximum length of the ``input_values`` of the returned list and optionally padding length (see above).
        max_length_labels (:obj:`int`, `optional`):
            Maximum length of the ``labels`` returned list and optionally padding length (see above).
        pad_to_multiple_of (:obj:`int`, `optional`):
            If set will pad the sequence to a multiple of the provided value.
            This is especially useful to enable the use of Tensor Cores on NVIDIA hardware with compute capability >=
            7.5 (Volta).
    """

    processor: Wav2Vec2Processor
    padding: Union[bool, str] = True
    max_length: Optional[int] = None
    max_length_labels: Optional[int] = None
    pad_to_multiple_of: Optional[int] = None
    pad_to_multiple_of_labels: Optional[int] = None

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # split inputs and labels since they have to be of different lenghts and need
        # different padding methods
        input_features = [{"input_values": feature["input_values"]} for feature in features]
        label_features = [{"input_ids": feature["labels"]} for feature in features]

        batch = self.processor.pad(
            input_features,
            padding=self.padding,
            max_length=self.max_length,
            pad_to_multiple_of=self.pad_to_multiple_of,
            return_tensors="pt",
        )
        with self.processor.as_target_processor():
            labels_batch = self.processor.pad(
                label_features,
                padding=self.padding,
                max_length=self.max_length_labels,
                pad_to_multiple_of=self.pad_to_multiple_of_labels,
                return_tensors="pt",
            )

        # replace padding with -100 to ignore loss correctly
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        batch["labels"] = labels

        return batch

## Initializing the Data Collator

We instantiate the `DataCollatorCTCWithPadding` using the previously defined processor.

This enables:
- Dynamic padding during training  
- Proper handling of variable-length audio inputs  
- Correct masking of padded labels for CTC loss  

The collator will now be used by the Trainer to efficiently construct training batches.

In [ ]:
data_collator = DataCollatorCTCWithPadding(processor=processor, padding=True)

##  Defining Evaluation Metric (WER)

We load the Word Error Rate (WER) metric for evaluating ASR performance.

WER measures transcription accuracy by computing:

(Substitutions + Deletions + Insertions) / Total Words

A lower WER indicates better model performance.

This metric provides a realistic assessment of speech-to-text quality.

In [ ]:
wer_metric = load_metric("wer")

The model will return a sequence of logit vectors:
$\mathbf{y}_1, \ldots, \mathbf{y}_m$ with $\mathbf{y}_1 = f_{\theta}(x_1, \ldots, x_n)[0]$ and $n >> m$.

A logit vector $\mathbf{y}_1$ contains the log-odds for each word in the vocabulary we defined earlier, thus $\text{len}(\mathbf{y}_i) =$ `config.vocab_size`. We are interested in the most likely prediction of the model and thus take the `argmax(...)` of the logits. Also, we transform the encoded labels back to the original string by replacing `-100` with the `pad_token_id` and decoding the ids while making sure that consecutive tokens are **not** grouped to the same token in CTC style ${}^1$.

## Implementing Evaluation Logic

We define a `compute_metrics` function to calculate Word Error Rate (WER) during evaluation.

This function:

- Converts model logits into predicted token IDs using `argmax`  
- Replaces `-100` (ignored label padding) with the tokenizer’s pad token  
- Decodes both predictions and reference labels into text  
- Computes WER between predicted and ground-truth transcripts  

This ensures accurate and consistent performance measurement throughout training.

In [ ]:
def compute_metrics(pred):
    pred_logits = pred.predictions
    pred_ids = np.argmax(pred_logits, axis=-1)

    pred.label_ids[pred.label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str = processor.batch_decode(pred_ids)
    # we do not want to group tokens when computing the metrics
    label_str = processor.batch_decode(pred.label_ids, group_tokens=False)

    wer = wer_metric.compute(predictions=pred_str, references=label_str)

    return {"wer": wer}

##  Loading and Configuring the Pretrained XLSR Model

We load the pretrained `facebook/wav2vec2-large-xlsr-53` model and adapt it for our ASR task.

Key configurations:

- Dropout parameters (`attention_dropout`, `hidden_dropout`, `layerdrop`) improve regularization  
- `mask_time_prob` enables time masking for robustness  
- `gradient_checkpointing=True` reduces memory usage during training  
- `ctc_loss_reduction="mean"` ensures stable CTC optimization  
- `pad_token_id` aligns with our tokenizer  
- `vocab_size` is set to match our custom Punjabi vocabulary  

This step transfers multilingual speech representations into a task-specific ASR model ready for fine-tuning.

In [ ]:
from transformers import Wav2Vec2ForCTC

model = Wav2Vec2ForCTC.from_pretrained(
    "facebook/wav2vec2-large-xlsr-53",
    attention_dropout=0.1,
    hidden_dropout=0.1,
    feat_proj_dropout=0.0,
    mask_time_prob=0.05,
    layerdrop=0.1,
    gradient_checkpointing=True,
    ctc_loss_reduction="mean",
    pad_token_id=processor.tokenizer.pad_token_id,
    vocab_size=len(processor.tokenizer)
)

## Freezing the Feature Extractor

We freeze the convolutional feature encoder layers of the pretrained model.

Why?

- These layers already capture general acoustic representations  
- Prevents overfitting on limited training data  
- Stabilizes fine-tuning  
- Reduces computational cost  

This allows the higher Transformer layers to adapt specifically to our target language while retaining robust low-level speech features.

In [ ]:
model.freeze_feature_extractor()

## Configuring Training Arguments

We define the training configuration using Hugging Face `TrainingArguments`.

Key settings:

- `per_device_train_batch_size=16` with gradient accumulation for effective larger batches  
- `num_train_epochs=200` for extensive fine-tuning  
- `learning_rate=3e-4` with `warmup_steps=500` for stable optimization  
- `fp16=True` enables mixed precision for faster training  
- `evaluation_strategy="steps"` for periodic validation  
- Checkpointing and logging every 500 steps  

These hyperparameters control optimization, evaluation frequency, memory efficiency, and overall training stability.

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
  output_dir="/content/gdrive/MyDrive/wav2vec2-large-xlsr-pa-IN",
  # output_dir="./wav2vec2-large-xlsr-turkish-demo",
  group_by_length=True,
  per_device_train_batch_size=16,
  gradient_accumulation_steps=2,
  evaluation_strategy="steps",
  num_train_epochs=200,
  fp16=True,
  save_steps=500,
  eval_steps=500,
  logging_steps=500,
  learning_rate=3e-4,
  warmup_steps=500,
  save_total_limit=2,
)

## Initializing the Trainer

We instantiate the Hugging Face `Trainer` to manage the fine-tuning process.

The Trainer integrates:

- The pretrained XLSR model  
- Custom data collator for dynamic padding  
- Training configuration (hyperparameters)  
- Evaluation metric (WER)  
- Prepared training and test datasets  

This abstracts the training loop, handling forward pass, backpropagation, evaluation, logging, and checkpointing in a structured and scalable way.

In [ ]:
from transformers import Trainer
trainer = Trainer(
    model=model,
    data_collator=data_collator,
    args=training_args,
    compute_metrics=compute_metrics,
    train_dataset=common_voice_train,
    eval_dataset=common_voice_test,
    tokenizer=processor.feature_extractor,
)

##  Preventing Colab Runtime Disconnection

This JavaScript snippet automatically clicks the “Connect” button every 60 seconds to prevent the Colab session from timing out.

Long ASR training jobs can take several hours, and runtime disconnections may interrupt training.  
This ensures continuous execution during extended fine-tuning sessions.

### Training

```javascript
function ConnectButton(){
    console.log("Connect pushed");
    document.querySelector("#top-toolbar > colab-connect-button").shadowRoot.querySelector("#connect").click()
}
setInterval(ConnectButton,60000);
```

##  Starting Model Fine-Tuning

We initiate the training process using the configured `Trainer`.

During training:

- Audio inputs are passed through the pretrained XLSR encoder  
- Transformer layers adapt to the target language  
- CTC loss aligns predicted tokens with transcripts  
- Model weights are updated via backpropagation  

Training logs, evaluation metrics (WER), and checkpoints are generated periodically based on the defined training arguments.

This step fine-tunes the multilingual model into a task-specific ASR system.

In [ ]:
trainer.train()

##  Loading a Saved Checkpoint for Inference

We reload a previously saved checkpoint from Google Drive and move the model to GPU for inference.

- The fine-tuned model weights are restored from `checkpoint-1500`  
- The processor is reloaded to ensure consistent preprocessing  
- The model is moved to `"cuda"` for accelerated computation  

This allows us to resume training or perform evaluation/inference using the saved ASR model.

In [ ]:
model = Wav2Vec2ForCTC.from_pretrained("/content/gdrive/MyDrive/wav2vec2-large-xlsr-pa-IN/checkpoint-1500").to("cuda")
processor = Wav2Vec2Processor.from_pretrained("/content/gdrive/MyDrive/wav2vec2-large-xlsr-pa-IN")

## Running Inference on a Test Sample

We perform inference on a test audio sample:

- Process the audio input using the saved processor  
- Pass the input through the fine-tuned model on GPU  
- Obtain output logits  
- Select the most probable token IDs using `argmax`  

This generates raw predicted token IDs, which will be decoded into readable text in the next step.

In [ ]:
input_dict = processor(common_voice_test["input_values"][0], return_tensors="pt", padding=True)

logits = model(input_dict.input_values.to("cuda")).logits

pred_ids = torch.argmax(logits, dim=-1)[0]

##  Reloading Test Dataset for Reference Transcripts

We reload the Punjabi test split of the Common Voice dataset to access the original ground-truth transcripts.

This allows us to:
- Compare model predictions with actual transcripts  
- Manually inspect transcription quality  
- Validate inference performance  

Having clean reference data is essential for qualitative evaluation.

In [ ]:
common_voice_test_transcription = load_dataset("common_voice", "pa-IN", data_dir="./cv-corpus-6.1-2020-12-11", split="test")

Finally, we can decode the example.

In [ ]:
print("Prediction:")
print(processor.decode(pred_ids))

print("\nReference:")
print(common_voice_test_transcription["sentence"][0].lower())